In [18]:
from multiprocessing import context

from dask.tests.test_delayed import modlevel_eager
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

EMBED_MODEL = "text-embedding-qwen3-embedding-8b"
LMSTUIO_BASE_URL = "http://host.docker.internal:12345/v1"
LMSTUDIO_API_KEY = "lm-studio"

embedding_function = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUIO_BASE_URL,
    api_key = LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False, # 토큰 ID 대신 원문 문자열 전송
)

_v = embedding_function.embed_query("연결 테스트")
print("임베딩 차원:", len(_v))

임베딩 차원: 4096


In [19]:
import chromadb

CHROMA_HOST ="chromadb"
CHROMA_PORT = 8000

chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)

print("heartbeat:", chroma_client.heartbeat())

heartbeat: 1782359501682441464


In [20]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

COLLECTION = "kb_realestate_2024"
PDF_PATH = "2024_KB_부동산_보고서_최종.pdf"

vectorstore = Chroma(
    client=chroma_client,
    collection_name=COLLECTION,
    embedding_function=embedding_function,
)

count = vectorstore._collection.count()
if count == 0:
    print("컬렉션이 비어 있음 -> 인덱싱 시작")
    documents = PyPDFLeader(PDF_PATH).load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)
    print("분할된 청크 수:", len(chunks))
    vectorstore.add_documents(chunks)
    print("적재 완료. 문서 수:", vectorstore._collection.count())
else:
    print("이미 적재됨. 문서 수:", count, "-> 인덱싱 건너뜀")

이미 적재됨. 문서 수: 215 -> 인덱싱 건너뜀


In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_openai.chat_models.base import ChatOpenAI
from langchain_community.chat_message_histories import ChatMessageHistory

LLM_MODEL = "gemma-4-e2b-it"
retriever = vectorstore.as_retriever(search_kwargs={"k":3})

template = """당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

컨텍스트: {context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("placeholder", "{chat_history}"),
    ("human", "{question}"),
])

model = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUIO_BASE_URL,
    api_key="lmstudio",
    temperature=0.7,
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
base_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | model
    | StrOutputParser()
)

store = {}





def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain = RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

In [24]:
resp = chain.invoke(
    {"question": "수도권 주택 매매 전망을 알려줘"},
    {"configurable": {"session_id": "nb-test"}},
)

print(resp)

KB 부동산 보고서 전문가로서 현재 수도권 주택 시장에 대한 전반적인 전망과 주요 특징을 종합하여 상세하게 설명해 드리겠습니다.

---

## 📊 수도권 주택 매매 전망 요약

현재 수도권 주택 시장은 **'높은 금리 부담으로 인한 전반적인 침체 기조 속에서, 지역별 특성과 규제 완화 기대감이 복합적으로 작용하는 양상'**을 보이고 있습니다.

### 1. 핵심 동인 (Key Drivers)

| 구분 | 내용 | 영향 |
| :--- | :--- | :--- |
| **금리 부담** | 높은 기준금리가 가장 큰 매수자들의 구매력 저하 요인입니다. 이자 부담이 주택 매매를 망설이게 하는 핵심 이유로 작용하고 있습니다. | **매매가격 하락 압력**의 주요 원인 |
| **규제 영향** | DSR 규제 등 대출 규제는 매수자의 구매 여력을 제한하고 있습니다. | 시장 전반의 거래량 위축 |
| **매도자 심리 개선** | 2023년 이후 정부의 다양한 규제 완화 조치로 인해 **매도자들의 기대 심리는 높아지고 있습니다.** (재건축 규제 완화, GTX 등 호재 기대) | 일부 지역 및 특정 상품에 대한 **상승 기대감** 형성 |

### 2. 가격 전망: 하락 우위 vs. 지역별 차별화

#### 📉 매매가격 측면: 하락 전망 우세
* **전반적 하락 우세:** 부동산 시장 전문가, 공인중개사, PB들의 설문 조사 결과에 따르면, **2024년 전국 주택 매매가격은 지난해에 이어 올해도 하락세가 이어질 것**이라는 전망이 우세합니다.
* **하락의 주요 원인:** 금리에 따른 이자 부담과 전반적인 경기 둔화가 가격 하락을 주도할 것으로 예상됩니다.

#### 🏘️ 지역별 차별화 (선호 지역 vs. 잠재적 위험 지역)
시장은 '전국'이 아닌 '지역' 단위로 움직이고 있습니다.

* **강남권 (한강이남):** 정부의 규제 완화와 주택 수요 집중으로 인해 **상대적인 강세 기대감**을 유지하고 있습니다. 높은 분양가에도 불구하고 청약 수요는 꾸준하며, 정비사업의

In [25]:
%%writefile app.py
import os
import chromadb
import streamlit as st

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
import langchain_community.document_loaders import PyPDFLoader
import langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory


LMSTUIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://host.docker.internal:12345/v1")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-qwen3-embedding-8b")
LLM_MODEL = os.getenv("LLM_MODEL", "gemma-4-e2b-it")
LMSTUDIO_API_KEY = os.getenv("LMSTUDIO_API_KEY", "lm-studio")
CHROMA_HOST = os.getenv("CHROMA_HOST", "chromadb")
CHROMA_PORT = int(os.getenv("CHROMA_PORT", "8000"))
COLLECTION = os.getenv("COLLECTION", "kb_realestate_2024")
PDF_PATH = os.getenv("PDF_PATH", "./2024_KB_부동산_보고서_최종.pdf")

@st.cache_resource
def get_embeddings():
    return OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=LMSTUIO_BASE_URL,
        api_key=LMSTUDIO_API_KEY,
        check_embedding_ctx_length=False,
    )

@st.cache_resource
def initialize_vectorstore():
    client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    vs = Chroma(client=client, collection_name=COLLECTION, embedding_function=get_embeddings())
    if vs._collection.count() == 0:
        documents = PyPDFLoader(PDF_PATH).load()
        splitter = RecursiveCharacterTextSplitter(chuck_size=1000, chuck_overlap=200)
        vs.add_documents(splitter.split_documents(documents))
    return vs

@st.cache_resource
def initialize_chain():
    vectorstore = initialize_vectorstore()
    retriever = vectorstor.as_retriever(search_kwargs={"k":3})

    template = """당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해부세요.

    컨텍스트: {context}
    """

    prompt = ChatPromptTemplate.from_messages([
        ("system", template),
        ("placeholder", "{chat_history}"),
        ("human", "{question}"),
    ])

    model = ChatOpenAI(
        model=LLM_MODEL,
        base_url=LMSTUIO_BASE_URL,
        api_key=LMSTUDIO_API_KEY,
        temperature=0.7,
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    base_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["question"]))
        )
        | prompt
        | model
        | StrOutputParser()
    )

    store = {}

    def get_history(session_id: str):
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]

    return RunnableWithMessageHistory(
        base_chain,
        get_history,
        input_messages_key="question",
        history_messages_key="chat_history",
    )

def main():
    st.set_page_config(page_title="KB 부동산 보고서 챗봇", page_icon="\U0001F3E0")
    st.title("\U0001F3E0 KB 부동산 보고서 AI 어드바이저")
    st.caption("2024 KB 부동산 보고서 기반 질의응답 시스템")

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    if prompt := st.chat_input("부동산 관련 질문을 입력하세요"):
        with st.chat_message("user"):
            st.markdown(prompt)
        st.session_state.messages.append({"role":"user", "contest": prompt})

        chain = initialize_chain()
        with st.chat_message("assistant"):
            with st.spinner("답변 생성 중..."):
                response = chain.invoke(
                    {"question": prompt},
                    {"configurable": {"session_id": "streamlit_session"}},
                )
            st.markdown(response)
        st.session_state.messages.append({"role":"assistant", "content": response})

if __name__ == "__main__":
    main()

Writing app.py
